# Task 07 — Health endpoint

**Wave 1** · target file: `backend/main.py` · prerequisites: task 01, task 06

**🎯 Goal:** FastAPI `app` + lifespan + CORS + `GET /`.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [1]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys

p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)

os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)

os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')

print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

repo root: /home/joseb/code/zhhrozhh/Macrocosm
models: ['baseline_stack.pkl', 'fake_image_model.pkl']


## What to build
In `backend/main.py`:
- an **async `lifespan`** handler that calls `load_models()` then `yield`,
- **`app = FastAPI(title=settings.TITLE, lifespan=lifespan)`**,
- **CORS**: `app.add_middleware(CORSMiddleware, allow_origins=settings.CORS_ORIGINS, allow_methods=["*"], allow_headers=["*"])`,
- **`GET "/"`** returning `{"status":"ok", "tabular_model": <baseline class name>, "image_model": <image class name>, "input_shape": settings.IMG_SHAPE}`.

> 10 will add `POST /predict` to this same file. Implement in `backend/main.py`, then run **Check**.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [2]:
# You write the app in backend/main.py (it can't be exercised from this cell). Plan:
#   1. async lifespan -> load_models() then yield
#   2. app = FastAPI(title=settings.TITLE, lifespan=lifespan)
#   3. app.add_middleware(CORSMiddleware, allow_origins=settings.CORS_ORIGINS, allow_methods=["*"], allow_headers=["*"])
#   4. @app.get("/") -> {"status":"ok","tabular_model":..., "image_model":..., "input_shape": settings.IMG_SHAPE}
print("implement the app + GET / in backend/main.py, then run Check")

implement the app + GET / in backend/main.py, then run Check


## ➡️ Move it into `backend/main.py`
Once the cell above passes, put your implementation into **`backend/main.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [6]:
import os, joblib, numpy as np, tempfile, importlib
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30)), "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")
os.environ["BASELINE_PATH"] = f"{TMP}/b.pkl"; os.environ["IMAGE_MODEL_PATH"] = f"{TMP}/i.pkl"
import backend.config, backend.model, backend.main
for m in (backend.config, backend.model, backend.main): importlib.reload(m)
from fastapi.testclient import TestClient
r = TestClient(backend.main.app).get("/")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("health OK:", r.json())

/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWar

health OK: {'status': 'ok', 'tabular_model': 'StackingRegressor', 'image_model': 'RandomRedshiftModel', 'input_shape': [64, 64, 5]}


/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StackingRegressor from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


> ⚠️ **07 + 10 both edit `backend/main.py`.** 07 creates the app + `GET /`; 10 adds `POST /predict`. Keep *both* — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/07-health-endpoint
!git add backend/main.py notebooks/tasks-2026-6-17/07-health-endpoint/task.ipynb
!git commit -m "07-health-endpoint: Health endpoint"
!git push -u origin task/07-health-endpoint
# then merge back into 2026.6.17 -> see MERGE.md in this folder